# 04 · Pipeline — End to End
**Purpose:** Run the complete ETL pipeline in a single notebook, exactly as 
the Airflow DAG would execute it — bootstrap → extract → transform → load → audit.

This is the notebook to run in GitHub Codespaces for a quick end-to-end test.

---
### Pipeline topology
```
bootstrap_schema
      │
      ├── extract coin prices  ──┐
      │                         ├── load to SQLite → log_run_complete
      └── extract global stats ─┘
```


In [ ]:
# ── 0. Dependencies ─────────────────────────────────────────────
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "requests", "-q"])

import os, sqlite3, logging, uuid, requests
from datetime import datetime, timezone
from contextlib import contextmanager

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%H:%M:%S"
)
logger = logging.getLogger(__name__)

SQLITE_PATH = "data/crypto_market.db"
BASE_URL    = "https://api.coingecko.com/api/v3"
COINS = [
    "bitcoin", "ethereum", "solana", "cardano",
    "polkadot", "chainlink", "avalanche-2", "uniswap"
]
os.makedirs("data", exist_ok=True)
print("✓ Setup complete")


In [ ]:
# ── 1. Database helpers ──────────────────────────────────────────

@contextmanager
def get_connection():
    conn = sqlite3.connect(SQLITE_PATH)
    conn.row_factory = sqlite3.Row
    try:
        yield conn
        conn.commit()
    except Exception:
        conn.rollback()
        raise
    finally:
        conn.close()

def bootstrap_schema():
    with get_connection() as conn:
        cur = conn.cursor()
        cur.execute("""CREATE TABLE IF NOT EXISTS coin_prices (
            id INTEGER PRIMARY KEY AUTOINCREMENT, coin_id TEXT NOT NULL,
            symbol TEXT NOT NULL, name TEXT NOT NULL, current_price_usd REAL,
            market_cap_usd INTEGER, market_cap_rank INTEGER,
            fully_diluted_valuation_usd INTEGER, total_volume_24h_usd INTEGER,
            high_24h_usd REAL, low_24h_usd REAL, price_change_24h_usd REAL,
            price_change_pct_1h REAL, price_change_pct_24h REAL, price_change_pct_7d REAL,
            volatility_score REAL, circulating_supply REAL, total_supply REAL,
            max_supply REAL, ath_usd REAL, ath_change_pct REAL, atl_usd REAL,
            last_updated TEXT, ingested_at TEXT NOT NULL, UNIQUE(coin_id, ingested_at))""")
        cur.execute("""CREATE TABLE IF NOT EXISTS global_market_stats (
            id INTEGER PRIMARY KEY AUTOINCREMENT, total_market_cap_usd INTEGER,
            total_volume_24h_usd INTEGER, btc_dominance_pct REAL, eth_dominance_pct REAL,
            active_cryptocurrencies INTEGER, total_exchanges INTEGER,
            market_cap_change_pct_24h REAL, ingested_at TEXT NOT NULL, UNIQUE(ingested_at))""")
        cur.execute("""CREATE TABLE IF NOT EXISTS pipeline_run_log (
            id INTEGER PRIMARY KEY AUTOINCREMENT, run_id TEXT NOT NULL,
            started_at TEXT NOT NULL, completed_at TEXT, records_loaded INTEGER DEFAULT 0,
            status TEXT DEFAULT 'RUNNING', error_message TEXT)""")
        for idx in [
            "CREATE INDEX IF NOT EXISTS idx_cp_coin ON coin_prices(coin_id)",
            "CREATE INDEX IF NOT EXISTS idx_cp_ts ON coin_prices(ingested_at)",
        ]:
            cur.execute(idx)
    logger.info("✓ Schema ready")

print("✓ DB helpers defined")


In [ ]:
# ── 2. Extract ───────────────────────────────────────────────────

def extract_market_data():
    logger.info(f"Extracting market data for {len(COINS)} coins...")
    params = {
        "vs_currency": "usd", "ids": ",".join(COINS),
        "order": "market_cap_desc", "per_page": len(COINS), "page": 1,
        "sparkline": "false", "price_change_percentage": "1h,24h,7d"
    }
    r = requests.get(f"{BASE_URL}/coins/markets", params=params, timeout=15)
    r.raise_for_status()
    data = r.json()
    logger.info(f"✓ Extracted {len(data)} coin records")
    return data

def extract_global_stats():
    logger.info("Extracting global market stats...")
    r = requests.get(f"{BASE_URL}/global", timeout=15)
    r.raise_for_status()
    data = r.json().get("data", {})
    logger.info("✓ Extracted global stats")
    return data

print("✓ Extract functions defined")


In [ ]:
# ── 3. Transform ─────────────────────────────────────────────────

def transform_market_data(raw_records):
    ingested_at = datetime.now(timezone.utc).isoformat()
    result = []
    for r in raw_records:
        try:
            p1h  = r.get("price_change_percentage_1h_in_currency")  or 0.0
            p24h = r.get("price_change_percentage_24h_in_currency") or 0.0
            p7d  = r.get("price_change_percentage_7d_in_currency")  or 0.0
            lu   = r.get("last_updated", ingested_at)
            if lu:
                lu = datetime.fromisoformat(lu.replace("Z", "+00:00")).isoformat()
            result.append({
                "coin_id": r.get("id","unknown"), "symbol": (r.get("symbol") or "").upper(),
                "name": r.get("name","Unknown"),
                "current_price_usd": round(float(r.get("current_price") or 0), 6),
                "market_cap_usd": int(r.get("market_cap") or 0),
                "market_cap_rank": r.get("market_cap_rank"),
                "fully_diluted_valuation_usd": r.get("fully_diluted_valuation"),
                "total_volume_24h_usd": int(r.get("total_volume") or 0),
                "high_24h_usd": round(float(r.get("high_24h") or 0), 6),
                "low_24h_usd": round(float(r.get("low_24h") or 0), 6),
                "price_change_24h_usd": round(float(r.get("price_change_24h") or 0), 6),
                "price_change_pct_1h": round(p1h, 4), "price_change_pct_24h": round(p24h, 4),
                "price_change_pct_7d": round(p7d, 4),
                "volatility_score": round(abs(p1h - p24h), 4),
                "circulating_supply": r.get("circulating_supply"),
                "total_supply": r.get("total_supply"), "max_supply": r.get("max_supply"),
                "ath_usd": round(float(r.get("ath") or 0), 6),
                "ath_change_pct": round(float(r.get("ath_change_percentage") or 0), 4),
                "atl_usd": round(float(r.get("atl") or 0), 10),
                "last_updated": lu, "ingested_at": ingested_at,
            })
        except (TypeError, ValueError, KeyError) as e:
            logger.warning(f"Skipping {r.get('id','?')}: {e}")
    logger.info(f"✓ Transformed {len(result)}/{len(raw_records)} records")
    return result

def transform_global_stats(raw):
    if not raw:
        return None
    mc  = raw.get("total_market_cap", {})
    vol = raw.get("total_volume", {})
    pct = raw.get("market_cap_percentage", {})
    return {
        "total_market_cap_usd":      int(mc.get("usd") or 0),
        "total_volume_24h_usd":      int(vol.get("usd") or 0),
        "btc_dominance_pct":         round(float(pct.get("btc") or 0), 4),
        "eth_dominance_pct":         round(float(pct.get("eth") or 0), 4),
        "active_cryptocurrencies":   raw.get("active_cryptocurrencies"),
        "total_exchanges":           raw.get("markets"),
        "market_cap_change_pct_24h": round(float(raw.get("market_cap_change_percentage_24h_usd") or 0), 4),
        "ingested_at":               datetime.now(timezone.utc).isoformat(),
    }

print("✓ Transform functions defined")


In [ ]:
# ── 4. Load ──────────────────────────────────────────────────────

def load_coin_prices(records):
    if not records:
        return 0
    cols = list(records[0].keys())
    sql  = f"INSERT OR IGNORE INTO coin_prices ({', '.join(cols)}) VALUES ({', '.join(['?']*len(cols))})"
    with get_connection() as conn:
        conn.executemany(sql, [tuple(r[c] for c in cols) for r in records])
    logger.info(f"✓ Loaded {len(records)} coin price records")
    return len(records)

def load_global_stats(stats):
    if not stats:
        return 0
    cols = list(stats.keys())
    sql  = f"INSERT OR IGNORE INTO global_market_stats ({', '.join(cols)}) VALUES ({', '.join(['?']*len(cols))})"
    with get_connection() as conn:
        conn.execute(sql, tuple(stats[c] for c in cols))
    logger.info("✓ Loaded global stats")
    return 1

def log_run(run_id, status, records_loaded=0, error=None):
    now = datetime.now(timezone.utc).isoformat()
    with get_connection() as conn:
        conn.execute(
            "INSERT INTO pipeline_run_log (run_id, started_at, completed_at, records_loaded, status, error_message) VALUES (?,?,?,?,?,?)",
            (run_id, now, now, records_loaded, status, error)
        )
    logger.info(f"✓ Run logged: {run_id} → {status}")

print("✓ Load functions defined")


---
## 🚀 Run the full pipeline
Execute the cell below to trigger one complete ETL run, exactly like the Airflow DAG would.


In [ ]:
RUN_ID = str(uuid.uuid4())[:8]
started = datetime.now(timezone.utc)
coin_records_loaded = 0

print(f"{'='*55}")
print(f"  Pipeline run: {RUN_ID}")
print(f"  Started at : {started.strftime('%Y-%m-%d %H:%M:%S UTC')}")
print(f"{'='*55}\n")

try:
    # Task 1 — bootstrap
    print("[1/4] Bootstrapping schema...")
    bootstrap_schema()

    # Task 2 — coin prices ETL (parallel in Airflow, sequential here)
    print("\n[2/4] Coin prices ETL...")
    raw_coins   = extract_market_data()
    clean_coins = transform_market_data(raw_coins)
    coin_records_loaded = load_coin_prices(clean_coins)

    # Task 3 — global stats ETL
    print("\n[3/4] Global stats ETL...")
    raw_global   = extract_global_stats()
    clean_global = transform_global_stats(raw_global)
    load_global_stats(clean_global)

    # Task 4 — audit log
    print("\n[4/4] Writing run log...")
    log_run(RUN_ID, "SUCCESS", records_loaded=coin_records_loaded)

    elapsed = (datetime.now(timezone.utc) - started).total_seconds()
    print(f"\n{'='*55}")
    print(f"  ✅ Pipeline complete in {elapsed:.2f}s")
    print(f"  Coin records : {coin_records_loaded}")
    print(f"  DB path      : {os.path.abspath(SQLITE_PATH)}")
    print(f"{'='*55}")

except Exception as e:
    log_run(RUN_ID, "FAILED", error=str(e))
    logger.error(f"Pipeline failed: {e}")
    raise


---
## Results


In [ ]:
# ── Current snapshot ─────────────────────────────────────────────
with get_connection() as conn:
    rows = conn.execute("""
        SELECT coin_id, symbol, current_price_usd, price_change_pct_1h,
               price_change_pct_24h, price_change_pct_7d, volatility_score, market_cap_rank
        FROM coin_prices
        WHERE ingested_at = (SELECT MAX(ingested_at) FROM coin_prices)
        ORDER BY market_cap_rank
    """).fetchall()

    global_row = conn.execute(
        "SELECT * FROM global_market_stats ORDER BY ingested_at DESC LIMIT 1"
    ).fetchone()

print(f"\n{'#':<4} {'Coin':<14} {'Price':>14} {'1h':>7} {'24h':>7} {'7d':>7} {'Volatility':>12}")
print("-" * 68)
for r in rows:
    def fmt_pct(v):
        return f"{'▲' if v >= 0 else '▼'}{abs(v):.2f}%"
    print(f"  {r['market_cap_rank']:<3} {r['coin_id']:<14} "
          f"${r['current_price_usd']:>13,.4f} "
          f"{fmt_pct(r['price_change_pct_1h']):>7} "
          f"{fmt_pct(r['price_change_pct_24h']):>7} "
          f"{fmt_pct(r['price_change_pct_7d']):>7} "
          f"{r['volatility_score']:>12.4f}")

print(f"\n{'Global Market Stats':}")
print("-" * 40)
if global_row:
    print(f"  Total market cap : ${global_row['total_market_cap_usd']:>20,.0f}")
    print(f"  24h volume       : ${global_row['total_volume_24h_usd']:>20,.0f}")
    print(f"  BTC dominance    : {global_row['btc_dominance_pct']:>19.2f}%")
    print(f"  ETH dominance    : {global_row['eth_dominance_pct']:>19.2f}%")
    print(f"  Active coins     : {global_row['active_cryptocurrencies']:>20,}")


In [ ]:
# ── Run log ──────────────────────────────────────────────────────
with get_connection() as conn:
    log_rows = conn.execute("""
        SELECT run_id, status, records_loaded, started_at
        FROM pipeline_run_log ORDER BY started_at DESC LIMIT 10
    """).fetchall()

print(f"Pipeline run history (last {len(log_rows)} runs):")
print(f"{'Run ID':<10} {'Status':<10} {'Records':>8}  Timestamp")
print("-" * 55)
for r in log_rows:
    icon = "✅" if r["status"] == "SUCCESS" else "❌"
    print(f"  {icon} {r['run_id']:<8} {r['status']:<10} {r['records_loaded']:>8}  {r['started_at']}")


---
## ▶ Re-run anytime
Run the **🚀 Run the full pipeline** cell again at any time.  
The `INSERT OR IGNORE` upsert strategy means re-runs within the same minute  
will skip duplicates — the `(coin_id, ingested_at)` unique constraint protects you.

In production this notebook's logic maps 1:1 to the Airflow DAG in `dags/crypto_market_etl_dag.py`.
